In [ ]:
import numpy as np
import pandas as pd
import os, sys
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

import xgboost as xgb

# Boston house prices with XGBoost (regression)

This is a copy of UCI ML housing dataset. https://archive.ics.uci.edu/ml/machine-learning-databases/housing/

This dataset was taken from the StatLib library which is maintained at Carnegie Mellon University.

The Boston house-price data of Harrison, D. and Rubinfeld, D.L. ‘Hedonic prices and the demand for clean air’, J. Environ. Economics & Management, vol.5, 81-102, 1978. Used in Belsley, Kuh & Welsch, ‘Regression diagnostics …’, Wiley, 1980. N.B. Various transformations are used in the table on pages 244-261 of the latter.

Source of project:
https://www.datacamp.com/community/tutorials/xgboost-in-python

In [ ]:
# California housing: a modern stand-in for the Boston dataset, which scikit-learn
# removed (its 'B' feature encoded race). Fetched once from sklearn, then cached.
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
data = housing.frame

In [ ]:
print("shape of the data:", data.shape)

In [ ]:
feature_names = housing.feature_names
feature_names

In [ ]:
# The last column is the target (median house value, in $100,000s); call it PRICE.
data = data.rename(columns={'MedHouseVal': 'PRICE'})
data.head()

In [ ]:
data.info()

#### If you have some missing values such as NA in the dataset you may or may not do a separate treatment for them, because XGBoost is capable of handling missing values internally.

In [ ]:
# Basic statistics of the data
data.describe()

In [ ]:
# Separate the target and rest of the variables
X, y = data.iloc[:,:-1],data.iloc[:,-1]

In [ ]:
# convert the dataset into an optimized data structure called Dmatrix
# that XGBoost supports and gives it acclaimed performance and efficiency gains
data_dmatrix = xgb.DMatrix(data=X,label=y)

## XGBoost's hyperparameters


- learning_rate: step size shrinkage used to prevent overfitting. Range is [0,1]
- max_depth: determines how deeply each tree is allowed to grow during any boosting round.
- subsample: percentage of samples used per tree. Low value can lead to underfitting.
- colsample_bytree: percentage of features used per tree. High value can lead to overfitting.
- n_estimators: number of trees you want to build.
- objective: determines the loss function to be used, e.g. reg:squarederror for regression problems, binary:hinge for classification that returns only a 0/1 decision, and binary:logistic for classification that returns a probability.

XGBoost also supports regularization parameters to penalize models as they become more complex and reduce them to simple (parsimonious) models.

- gamma: controls whether a given node will split based on the expected reduction in loss after the split. A higher value leads to fewer splits. Supported only for tree-based learners.
- alpha: L1 regularization on leaf weights. A large value leads to more regularization.
- lambda: L2 regularization on leaf weights and is smoother than L1 regularization.



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [ ]:
xg_reg = xgb.XGBRegressor(objective ='reg:squarederror', colsample_bytree = 0.3, learning_rate = 0.1,
                max_depth = 5, alpha = 10, n_estimators = 10)


In [ ]:
xg_reg.fit(X_train,y_train)

preds = xg_reg.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE: %f" % (rmse))

## k-fold Cross Validation using XGBoost

XGBoost supports k-fold cross validation via the cv() method. All you have to do is specify the nfolds parameter, which is the number of cross validation sets you want to build. Also, it supports many other parameters (check out this link) like:

 - num_boost_round: denotes the number of trees you build (analogous to n_estimators)
 - metrics: tells the evaluation metrics to be watched during CV
 - as_pandas: to return the results in a pandas DataFrame.
 - early_stopping_rounds: finishes training of the model early if the hold-out metric ("rmse" in our case) does not improve for a given number of rounds.
 - seed: for reproducibility of results.


In [ ]:
params = {"objective":"reg:squarederror",'colsample_bytree': 0.3,'learning_rate': 0.1,
                'max_depth': 5, 'alpha': 10}

cv_results = xgb.cv(dtrain=data_dmatrix, params=params, nfold=3,
                    num_boost_round=50,early_stopping_rounds=10,metrics="rmse", as_pandas=True, seed=123)

In [ ]:
cv_results.head()

In [ ]:
# final boosting round metric
print((cv_results["test-rmse-mean"]).tail(1))

### Visualize Boosting Trees and Feature Importance

You can also visualize individual trees from the fully boosted model that XGBoost creates using the entire housing dataset. XGBoost has a plot_tree() function that makes this type of visualization easy. Once you train a model using the XGBoost learning API, you can pass it to the plot_tree() function along with the number of trees you want to plot using the num_trees argument.

In [ ]:
xg_reg = xgb.train(params=params, dtrain=data_dmatrix, num_boost_round=10)

If this code throws the 'graphviz' error, consider installing the graphviz package via <b> pip install graphviz </b>. You may also need to run <b>sudo apt-get install graphviz </b> on command line

Another way to visualize your XGBoost models is to examine the importance of each feature column in the original dataset within the model.

One simple way of doing this involves counting the number of times each feature is split on across all boosting rounds (trees) in the model, and then visualizing the result as a bar graph, with the features ordered according to how many times they appear. XGBoost has a plot_importance() function that allows you to do exactly this.

In [ ]:
plt.rcParams['figure.figsize'] = [5, 5]
xgb.plot_importance(xg_reg)
plt.show()

# Detecting Parkinson’s Disease with XGBoost (classification)

Parkinson’s disease is a progressive disorder of the central nervous system affecting movement and inducing tremors and stiffness. It has 5 stages to it and affects more than 1 million individuals every year in India. This is chronic and has no cure yet. It is a neurodegenerative disorder affecting dopamine-producing neurons in the brain.

Source for this project:
https://data-flair.training/blogs/python-machine-learning-project-detecting-parkinson-disease/

### Data
The dataset was created by Max Little of the University of Oxford, in collaboration with the National Centre for Voice and Speech, Denver, Colorado, who recorded the speech signals. The original study published the feature extraction methods for general voice disorders.

This dataset is composed of a range of biomedical voice measurements from 31 people, 23 with Parkinson's disease (PD). Each column in the table is a particular voice measure, and each row corresponds one of 195 voice recording from these individuals ("name" column). The main aim of the data is to discriminate healthy people from those with PD, according to "status" column which is set to 0 for healthy and 1 for PD.

The data is in ASCII CSV format. The rows of the CSV file contain an instance corresponding to one voice recording. There are around six recordings per patient, the name of the patient is identified in the first column.For further information or to pass on comments, please contact Max Little (littlem '@' robots.ox.ac.uk). 

https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/

Max A. Little, Patrick E. McSharry, Eric J. Hunter, Lorraine O. Ramig (2008), 'Suitability of dysphonia measurements for telemonitoring of Parkinson's disease', IEEE Transactions on Biomedical Engineering (to appear).


In [ ]:
# Read the data to a Pandas dataframe
df = pd.read_csv(os.path.join("data", "parkinsons.data"))
print("shape of dataframe:", df.shape)
df.head()

### Attribute Information:

Matrix column entries (attributes):
- name - ASCII subject name and recording number
- MDVP:Fo(Hz) - Average vocal fundamental frequency
- MDVP:Fhi(Hz) - Maximum vocal fundamental frequency
- MDVP:Flo(Hz) - Minimum vocal fundamental frequency
- MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP - Several measures of variation in fundamental frequency
- MDVP:Shimmer,MDVP:Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,MDVP:APQ,Shimmer:DDA - Several measures of variation in amplitude
- NHR,HNR - Two measures of ratio of noise to tonal components in the voice
- status - Health status of the subject (one) - Parkinson's, (zero) - healthy
- RPDE,D2 - Two nonlinear dynamical complexity measures
- DFA - Signal fractal scaling exponent
- spread1,spread2,PPE - Three nonlinear measures of fundamental frequency variation 

In [ ]:
# Get the features and labels
features = df.drop(columns = ['status', 'name']).values

labels = df.loc[:,'status'].values

print("shape of features:", features.shape)
print("features")
print(features, "\n")

print("shape of labels:", labels.shape)
print("labels")
print(labels)

In [ ]:
# Get the count of each label (0 and 1) in labels
print("number of 1s in labels:", labels[labels==1].shape[0])
print("number of 0s in labels:", labels[labels==0].shape[0])

In [ ]:
# Split first, then scale, so the test set's range never leaks into training.
y = labels
x_train, x_test, y_train, y_test = train_test_split(features, y, test_size=0.2, random_state=7)

In [ ]:
# Fit the scaler on the training features only; apply the same transform to test.
scaler = MinMaxScaler((-1, 1))
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
# Train the model
model = xgb.XGBClassifier()
model.fit(x_train, y_train)

In [ ]:
# Calculate the accuracy on test set
y_pred = model.predict(x_test)
print("accuracy:", round(accuracy_score(y_test, y_pred)*100, 3))

# Labels are imbalanced (~75% are 1), so accuracy alone is misleading: always
# predicting 1 already scores ~75%. Report precision/recall/F1 and a baseline.
from sklearn.metrics import classification_report
from sklearn.dummy import DummyClassifier
print(classification_report(y_test, y_pred))
dummy = DummyClassifier(strategy='most_frequent').fit(x_train, y_train)
print("baseline accuracy:", round(dummy.score(x_test, y_test)*100, 3))